In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import time

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Build Model

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X, batch_y
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs, batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            correct_train += (predicted == batch_y).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X, batch_y
                outputs = model(batch_X)
                loss = loss_fn(outputs, batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct_test += (predicted == batch_y).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")
        
    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
    return history

In [ ]:
no_epochs = 20
batch_size = 64
learning_rates = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

In [ ]:
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super(FashionMNISTCNN, self).__init__()
        
        # Define the layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 3 * 3, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        # Apply layers and activations
        x = torch.relu(self.conv1(x))
        x = nn.MaxPool2d(2, 2)(x)
        x = torch.relu(self.conv2(x))
        x = nn.MaxPool2d(2, 2)(x)
        x = torch.relu(self.conv3(x))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
def create_dataloader(batch_size, train=True, device=device):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    dataset = datasets.FashionMNIST('~/.pytorch/F_MNIST_data/', download=True, train=train, transform=transform)
    data_list = []
    labels_list = []
    for data, label in dataset:
        data_list.append(data)
        labels_list.append(label)

    data_tensor = torch.stack(data_list).to(device)
    labels_tensor = torch.tensor(labels_list, dtype=torch.long).to(device)
    tensor_dataset = TensorDataset(data_tensor, labels_tensor)
    loader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=True, drop_)
    return loader

In [ ]:
train_loader = create_dataloader(batch_size, True, device)
test_loader = create_dataloader(64, False, device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

### Categorical Cross Entropy

In [ ]:
history_dict = dict()
for ilr in learning_rates:
    model = FashionMNISTCNN().to(device)
    optimizer = optim.SGD(model.parameters(), lr=ilr)
    history_dict[ilr] = train_model(model, train_loader, test_loader,
                                    loss_fn, optimizer, no_epochs)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']) * cycler('marker', ['^',',', '.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    val_acc_values = history_dict[ilr]['test_acc']
    ax.plot(epochs, val_acc_values, label = 'lr = '+str(ilr))

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend()
fig.savefig('TestMNISTLearningRate.png', dpi=300, bbox_inches='tight')